# 🩺 Pulsar Med AI: Colab Training & GGUF Export Pipeline
Repository: **[Kingman9407/pulsar-med-ai](https://github.com/Kingman9407/pulsar-med-ai)**

This notebook automatically clones your GitHub repository, downloads the base model from **Hugging Face**, fine-tunes on your clinical dataset, tests live inference outputs, and exports **`model.gguf`** for Flutter on-device deployment.

### 📥 Step 1: Clone or Pull from GitHub Repository
Clones `pulsar-med-ai` if running for the first time, or pulls the latest commits if already cloned.

In [ ]:
import os

REPO_URL = "https://github.com/Kingman9407/pulsar-med-ai.git"
REPO_DIR = "/content/pulsar-med-ai"

if not os.path.exists(REPO_DIR):
    print(f"[*] Cloning {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("[*] Repo exists. Pulling latest updates from GitHub...")
    %cd {REPO_DIR}
    !git pull

# Navigate directly into medtrain
%cd {REPO_DIR}/medtrain
print("[*] Current working directory:", os.getcwd())

### ⚡ Step 2: Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

### 📦 Step 3: Install Training Dependencies
Installs `transformers`, `peft`, `trl`, `accelerate`, `gguf`, etc.

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

### 🏥 Step 4: Verify Database & Schema Sanity
Tests the SQLite database schema and AI DSL action execution.

In [ ]:
!python test_db_and_ai.py

### 🚀 Step 5: Train Model & Export GGUF
Pulls `HuggingFaceTB/SmolLM2-135M-Instruct` from Hugging Face, fine-tunes on your dataset, merges adapters, and converts to `model.gguf`.

In [ ]:
!python train.py \
  --dataset_path categories/cat_01_patient_creation/data.jsonl \
  --epochs 3 \
  --batch_size 4 \
  --lr 2e-4 \
  --gguf_output ./model.gguf

### 🔍 Step 6: Test Model Inference & Display Generated Outputs
Runs unseen clinical patient registration prompts through the fine-tuned model and displays the generated `ADD_PATIENT` actions and conversational responses.

In [ ]:
!python test_inference.py --model_path ./medical_merged_model

### 💬 Step 7: Live Interactive Prompt Playground
Test any custom patient registration query live!

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = './medical_merged_model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map='auto')

def ask_model(user_query):
    prompt = (
        f"<|im_start|>system\nYou are a clinical AI assistant that parses consultations into database actions (ADD_PATIENT, LOG_VISIT, etc.) followed by a SAY: response.<|im_end|>\n"
        f"<|im_start|>user\n{user_query}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, temperature=0.2, pad_token_id=tokenizer.eos_token_id)
    
    print('USER QUERY:', user_query)
    print('MODEL OUTPUT:')
    print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

# Try custom test query:
ask_model('Please register new patient Daniel Craig, born 1968-03-02, male, blood group O negative.')

### 💾 Step 8: Download `model.gguf` to Local Mac
Downloads the generated GGUF model binary to place in `medical/assets/models/model.gguf`.

In [ ]:
import os
from google.colab import files

if os.path.exists('model.gguf'):
    mb = os.path.getsize('model.gguf') / (1024 * 1024)
    print(f'Downloading model.gguf ({mb:.2f} MB)...')
    files.download('model.gguf')
else:
    print('model.gguf not found. Please run Step 5 first.')